<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook02__Baseline_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Inference stack

So far it seems the red writing can be ignored if it says says something about numpy incompatibility

In [ ]:
!pip install pytorch-fid==0.3.0 open-clip-torch==2.24.0 pandas -q
print("Install complete.")
print("If Cell 3 throws a numpy ABI error: Runtime > Restart session, re-run Cell 1, skip Cell 2, then run Cell 3.")

In [ ]:

# === Boilerplate: Drive mount, working directory, GPU verification ===
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/MyDissertationCN6000'
os.chdir(PROJECT_ROOT)

# Silence HF download progress bars to prevent the GitHub-render widget bug
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
assert torch.cuda.is_available(), (
    "CUDA not available. Go to Runtime > Change runtime type and select an A100 or L4 GPU. "
    "If already set, reinstall PyTorch with CUDA: "
    "!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Importing json and path (to read prompts), SD pipeline

In [ ]:
import json
from pathlib import Path
from diffusers import StableDiffusionPipeline

Tested settings from notebook01 run

In [ ]:
SEEDS = [42, 1337, 2026, 7777, 2213935]
GUIDANCE = 9.0
STEPS = 50  # PNDM default — matches ESD/Ring-A-Bell papers
NEGATIVE = ("photograph, photo, realistic, 3d render, cgi, blurry, deformed face, "
            "extra fingers, ugly, low quality, oversaturated, anime, cartoon, "
            "watermark, signature, text")

Loading baseline SD v1.4

In [ ]:
from diffusers.pipelines.deepfloyd_if import safety_checker
pipe=StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")
pipe.set_progress_bar_config(disable=True)

In [ ]:
#testing
generator = torch.Generator("cuda").manual_seed(42)
image = pipe("a cat sitting on a windowsill", generator=generator, num_inference_steps=50).images[0]
image.save("cat_test.png")
display(image)

Generation

In [ ]:
def generate_set(prompts, seeds, output_dir):
  "Generate using one prompt per each seed, so 5 images per one prompt, with paired filename"
  output_dir.mkdir(parents=True, exist_ok=True)
  log=[]
  total=len(prompts)*len(seeds) #total generations equals 10 target, 50 unrelated times 5 seeds, it can also just be 300 but left like this just in case
  count=0
  #nested loop, generating using one prompt for each seed before continuing with another prompt
  for prompt_idx, prompt in enumerate(prompts):
    for seed in seeds:
      generator = torch.Generator("cuda").manual_seed(seed)
      image=pipe(
          prompt,
          negative_prompt=NEGATIVE,
          generator=generator,
          num_inference_steps=STEPS,
          guidance_scale=GUIDANCE,
      ).images[0]
      filename=f"p{prompt_idx:03d}_s{seed:06d}.png" #this will name the file a corresponding prompt + seed
      image.save(output_dir / filename)
      log.append({
          "prompt_idx": prompt_idx,
          "prompt": prompt,
          "seed": seed,
          "filename": filename,
      })
      count+=1
      if count%25==0 or count==total: #prints progress after every 25 images and at the end of each set
        print(f" {count}/{total}")
  return log

target = json.loads(Path("prompts/target_prompts.json").read_text())
unrelated = json.loads(Path("prompts/unrelated_prompts.json").read_text())

print(f"Generating {len(target) * len(SEEDS)} target images...")
log_t = generate_set(target, SEEDS, Path("outputs/baseline/target1"))

print(f"\nGenerating {len(unrelated) * len(SEEDS)} unrelated images...")
log_u = generate_set(unrelated, SEEDS, Path("outputs/baseline/unrelated1"))

# Save the seed log for reproducibility and Notebook 04's paired comparison
Path("logs").mkdir(exist_ok=True)
Path("logs/seeds_baseline1.json").write_text(
    json.dumps({"target": log_t, "unrelated": log_u}, indent=2)
)
print(f"\nSaved seed log to logs/seeds_baseline.json")
